# ED-Assistant Quickstart Notebook

This notebook demonstrates:
1) Importing the ED CSV dataset into the local SQLite database (`ed.sqlite`)
2) Querying `stay_summary(stay_id)` directly
3) Sending a message to the running FastAPI `/chat` endpoint

**Prerequisites:**
- Start Ollama in a separate terminal: `ollama serve`
- Start API in your project folder: `source .venv/bin/activate && uvicorn app.main:app --port 8000`
- Ensure your CSVs are in `data/ed/` (e.g., `data/ed/edstays.csv.gz`, etc.)


In [ ]:
# 0) Setup: change to your project directory if needed
import os, sys, pathlib
proj = pathlib.Path('~/Downloads/ed_assistant_fastapi').expanduser()
os.chdir(proj)
print('Working dir:', os.getcwd())

## 1) Import CSVs → SQLite
This runs the importer to create/update `ed.sqlite` from the files in `data/ed/`.

In [ ]:
from scripts.import_csv_to_db import import_all
import_all()
print('✅ Import complete')

## 2) Query a summary for a stay
First, we pick a valid `stay_id` from `edstays`, then call `stay_summary(stay_id)`.

In [ ]:
import sqlite3
from app.db_edcsv import get_conn
conn = get_conn()
row = conn.execute('SELECT stay_id FROM edstays LIMIT 1').fetchone()
stay_id = row['stay_id'] if row else None
conn.close()
stay_id

In [ ]:
from app.tools import stay_summary
summary = stay_summary(stay_id) if stay_id else None
summary

## 3) Talk to the running ED-Assistant API
Sends a message to `http://127.0.0.1:8000/chat`. Make sure your API is running in a terminal tab.

In [ ]:
import requests, json
payload = {
    'session_id': 'notebook-demo',
    'message': 'Ich warte seit 2 Stunden und habe Angst.',
    'patient_context': {'Alter': '68'}
}
r = requests.post('http://127.0.0.1:8000/chat', json=payload, timeout=60)
print(r.status_code)
print(json.dumps(r.json(), ensure_ascii=False, indent=2))

### Notes
- If `requests` is missing, run `pip install requests` in your venv.
- If `stay_id` is `None`, verify your CSVs are present under `data/ed/` and re-run the importer cell.